# Figure 3 — M8 pooling vs Fourier pooling localization convergence

This notebook generates **readme Figure 3**: a spatial comparison of **average pooling** vs **Fourier pooling** learning behaviour on the M8 single-view xyz localisation task.

It trains both branches on the **existing full M8 corpus** with the historical consensus learning rates from `GummyBearTomography_Final_Report.ipynb` / `M8_CANONICAL_LR_BY_ARCH` (`pooled=0.001`, `fourier=0.03`), logs predicted xyz on the **full filtered validation set** every epoch (including initialization), writes reusable JSON histories, then builds POV-Ray frames **only from those JSON files** (identity map: catalog millimetres = POV world, z-up).

Visual language: closer **face-on** translucent grey bear (same azimuth as the network-scene illustration, mostly +X); **green** (bright) validation targets; **deep red** average-pooling predictions + links; **dark blue** Fourier predictions + links (luminous, slightly dimmer than green).

Generated files go under gitignored `outputs/figure3_learning_convergence/` (not `figures/`).

Install from the repo root (no editable install): `pip install ".[dl]" -c requirements.txt`


In [1]:
from __future__ import annotations
from pathlib import Path
import shutil

# Install the library from the local repo (no editable install).

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

# Stale setuptools wheel dirs break reinstalls with "File exists: …dist-info".
_build = ROOT / "build"
if _build.exists():
    shutil.rmtree(_build)

# Figure 3 needs the ML stack; constrain pins via requirements.txt.
!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

# FEM optical generation (optional):
# !pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:

from gummybear.paths import display_path, repo_relative_path
from gummybear_illustration.figure3_export import (
    add_bear_to_figure3_scenes,
    add_loss_board_to_figure3_scenes,
    add_markers_to_figure3_scenes,
    figure3_epoch_list,
    generate_figure3_scenes,
    prepare_figure3_history,
    render_figure3_scenes,
)
from gummybear_illustration.figure3_history import (
    default_figure3_root,
    load_history,
    select_best_fourier_advantage_sample,
)
from gummybear_illustration.paths import repo_root
from tomography_ml.studies.single_view_m8 import M8_CANONICAL_LR_BY_ARCH

ROOT = repo_root()

# Gitignored: outputs/figure3_learning_convergence/ (JSON, POV, GIF, PNG).
OUTPUT_ROOT = default_figure3_root(ROOT)

# Full M8 protocol (matches final-report xyz study filters inside the package).
NUM_EPOCHS = 200
BATCH_SIZE = 32
SEED = 0
# None = entire filtered train + validation + test sets (per split).
N_TRACKED = None
DEVICE = None  # None → tomography_ml.get_device()

# If JSON histories already exist under OUTPUT_ROOT, skip retraining.
# Set FORCE_RETRAIN=True once after upgrading to log train/test-split samples too.
SKIP_TRAIN_IF_HISTORY_EXISTS = True
FORCE_RETRAIN = False

# Multi-bear layout: test (default), validation (left), train (in front of validation).
MAIN_BEAR_ID = "default"
VALIDATION_BEAR_ID = "validation"
VALIDATION_BEAR_SCALE = 0.38
# Shift in mesh spans (x, y, z). Negative x = left; +y = toward camera; +z lifts off floor.
VALIDATION_BEAR_SHIFT_SPANS = (-1.5, 1.5, 0.25)
TRAINING_BEAR_ID = "train"
TRAINING_BEAR_SCALE = 0.38
TRAINING_BEAR_SHIFT_SPANS = (-2.3, 1.5, 0.25)
# Loss board size and placement. Fractions are measured along camera forward/right/up.
BOARD_WIDTH_MM = 60.0
BOARD_ASPECT = 0.69
BOARD_THICKNESS_MM = 0.7
BOARD_CENTER_FWD_FRAC = 2
BOARD_CENTER_RIGHT_FRAC = -0.3
BOARD_CENTER_UP_FRAC = 0.1
# Axis end-tick FreeType size on the board PNG (texture is 3200×2200).
# Try ~120–200; larger values auto-shrink if labels would not fit.
BOARD_LABEL_FONT_PX = 160
# Pillow stroke width for the red/blue train-loss curves (None → ~0.054× texture width).
BOARD_CURVE_WIDTH_PX = 140
# Luminous epoch markers on the board face (None → ~2.5% of the shorter board edge).
# Keep small/discrete; intensity is soft in the POV builder.
BOARD_DOT_RADIUS_MM = 1.0

# Rendering (consumes JSON only; does not read live tensors).
RENDER = True
# Frame-level POV-Ray parallelism. True uses cpu_count-2 workers (at least 1).
RENDER_MULTIPROCESSING = True
# 1 = every epoch (full motion GIF). Increase for faster POV iteration.
RENDER_EPOCH_STRIDE = 1
GIF_DURATION_MS = 200

# POV camera in simulation millimetres (z-up). None = auto (network-scene
# azimuth, long standoff). Copy resolved values from the export printout
# to pin an exact view.
CAMERA_LOCATION = (-15.0,85.0,8.0)  # e.g. (90.0, -16.0, 28.0)
CAMERA_LOOK_AT = (-15.1, 0.5, 10.5)
CAMERA_FOV_DEG = 42.0
# Scales the scene key/fill lights only; marker glow is controlled separately.
LIGHT_INTENSITY = 1.35
# Per-ball predicted lights, only inside PRED_LUMINOSITY_HOLD_MM of the target.
BALL_LIGHTS = True
# Camera-aimed POV spotlights (False = omnidirectional point lights).
BALL_SPOTLIGHTS = True
BALL_SPOTLIGHT_ANGLE_DEG = 8.0
# Scales predicted-ball lights only (not marker emission, not scene key/fill).
# High enough that a hit reads as a local coloured patch on the bear.
BALL_LIGHT_INTENSITY = 8.0
# Green target emission (1/3 of the previous default).
GREEN_LUMINOSITY = 0
# Target→predicted cylinders (usually off).
DRAW_PREDICTION_LINKS = False
# Distance-based emission ramp on balls (usually off; use transparency instead).
DISTANCE_LUMINOSITY = True
# Distance-based transparency on predicted balls only (usually on).
DISTANCE_TRANSPARENCY = True
# Red emission when coincident with the green target (if DISTANCE_LUMINOSITY).
POOLING_LUMINOSITY_AT_TARGET = 0.5
# Blue emission when coincident with the green target (if DISTANCE_LUMINOSITY).
FOURIER_LUMINOSITY_AT_TARGET = 0.6
# Back-compat shared knob; leave unused unless you want one value for both.
PRED_LUMINOSITY_AT_TARGET = None
# Hold the on-target emission through this radius, then decay linearly.
PRED_LUMINOSITY_HOLD_MM = 1.0
# Emission reaches background at/beyond this distance.
PRED_LUMINOSITY_ZERO_MM = 2.5
# POV transmit when far (0=opaque, 1=invisible). On target uses PRED_TRANSMIT_AT_TARGET.
PRED_TRANSMIT_FAR = 0.9
PRED_TRANSMIT_AT_TARGET = 0.0
# Hold the on-target transparency through this radius, then decay linearly.
PRED_TRANSMIT_HOLD_MM = 2.0
# Transparency reaches PRED_TRANSMIT_FAR at/beyond this distance.
PRED_TRANSMIT_ZERO_MM = 5.0

print(f"ROOT={display_path(ROOT)}")
print(f"OUTPUT_ROOT={display_path(OUTPUT_ROOT)}")
print(f"M8_CANONICAL_LR_BY_ARCH={dict(M8_CANONICAL_LR_BY_ARCH)}")
print(
    f"NUM_EPOCHS={NUM_EPOCHS}  SEED={SEED}  N_TRACKED={N_TRACKED}  "
    f"SKIP_TRAIN_IF_HISTORY_EXISTS={SKIP_TRAIN_IF_HISTORY_EXISTS}  "
    f"RENDER={RENDER}  RENDER_MULTIPROCESSING={RENDER_MULTIPROCESSING}"
)
print(
    f"CAMERA_LOCATION={CAMERA_LOCATION}  CAMERA_LOOK_AT={CAMERA_LOOK_AT}  "
    f"CAMERA_FOV_DEG={CAMERA_FOV_DEG}  LIGHT_INTENSITY={LIGHT_INTENSITY}  "
    f"BALL_LIGHTS={BALL_LIGHTS}  BALL_SPOTLIGHTS={BALL_SPOTLIGHTS}  "
    f"BALL_SPOTLIGHT_ANGLE_DEG={BALL_SPOTLIGHT_ANGLE_DEG}  "
    f"BALL_LIGHT_INTENSITY={BALL_LIGHT_INTENSITY}  "
    f"GREEN_LUMINOSITY={GREEN_LUMINOSITY}  "
    f"DRAW_PREDICTION_LINKS={DRAW_PREDICTION_LINKS}  "
    f"DISTANCE_LUMINOSITY={DISTANCE_LUMINOSITY}  "
    f"DISTANCE_TRANSPARENCY={DISTANCE_TRANSPARENCY}  "
    f"POOLING_LUMINOSITY_AT_TARGET={POOLING_LUMINOSITY_AT_TARGET}  "
    f"FOURIER_LUMINOSITY_AT_TARGET={FOURIER_LUMINOSITY_AT_TARGET}  "
    f"PRED_LUMINOSITY_AT_TARGET={PRED_LUMINOSITY_AT_TARGET}  "
    f"PRED_LUMINOSITY_HOLD_MM={PRED_LUMINOSITY_HOLD_MM}  "
    f"PRED_LUMINOSITY_ZERO_MM={PRED_LUMINOSITY_ZERO_MM}  "
    f"PRED_TRANSMIT_FAR={PRED_TRANSMIT_FAR}  "
    f"PRED_TRANSMIT_AT_TARGET={PRED_TRANSMIT_AT_TARGET}  "
    f"PRED_TRANSMIT_HOLD_MM={PRED_TRANSMIT_HOLD_MM}  "
    f"PRED_TRANSMIT_ZERO_MM={PRED_TRANSMIT_ZERO_MM}"
)
print(
    "Coordinate convention: particle_x/y/z are simulation millimetres (z-up), "
    "identical to the STL / physical-scene POV world (identity transform)."
)


ROOT=.
OUTPUT_ROOT=outputs/figure3_learning_convergence
M8_CANONICAL_LR_BY_ARCH={'pooled': 0.001, 'fourier': 0.03, 'flatten': 0.0003}
NUM_EPOCHS=200  SEED=0  N_TRACKED=None  SKIP_TRAIN_IF_HISTORY_EXISTS=True  RENDER=True  RENDER_MULTIPROCESSING=True
CAMERA_LOCATION=(-15.0, 85.0, 8.0)  CAMERA_LOOK_AT=(-15.1, 0.5, 10.5)  CAMERA_FOV_DEG=42.0  LIGHT_INTENSITY=1.35  BALL_LIGHTS=True  BALL_SPOTLIGHTS=True  BALL_SPOTLIGHT_ANGLE_DEG=8.0  BALL_LIGHT_INTENSITY=8.0  GREEN_LUMINOSITY=0  DRAW_PREDICTION_LINKS=False  DISTANCE_LUMINOSITY=True  DISTANCE_TRANSPARENCY=True  POOLING_LUMINOSITY_AT_TARGET=0.5  FOURIER_LUMINOSITY_AT_TARGET=0.6  PRED_LUMINOSITY_AT_TARGET=None  PRED_LUMINOSITY_HOLD_MM=1.0  PRED_LUMINOSITY_ZERO_MM=2.5  PRED_TRANSMIT_FAR=0.9  PRED_TRANSMIT_AT_TARGET=0.0  PRED_TRANSMIT_HOLD_MM=2.0  PRED_TRANSMIT_ZERO_MM=5.0
Coordinate convention: particle_x/y/z are simulation millimetres (z-up), identical to the STL / physical-scene POV world (identity transform).


## 1. History, scene layers, render

Run the cells below **in order**. Training is skipped when JSON histories already exist (`SKIP_TRAIN_IF_HISTORY_EXISTS=True`).

1. **History** — train or reuse `prediction_history/*.json`
2. **Generate scene** — camera, sky, key/fill lights, floor (shared `figure3_world.inc` + stub `.pov` files)
3. **Add bear** — mesh include + instance (written once, then included by every epoch)
4. **Add markers** — per-epoch red/blue/green balls for a catalog **split** (`train` / `validation` / `test`) on a named **bear**. Call again for another split or bear; layers accumulate.
5. **Add loss board** — per-epoch POV-Ray chalkboard quad with log-train-MSE curves and current-epoch dots.
6. **Render** — POV-Ray frames, GIF, final still (`RENDER=True`)

Change camera knobs and re-run from **generate scene**. Change only the bear, markers, or board and re-run from that cell; the `.pov` files are rewritten to `#include` the updated layers.


In [3]:
layout, combined = prepare_figure3_history(
    repo_root_path=ROOT,
    output_root=OUTPUT_ROOT,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    n_tracked=N_TRACKED,
    skip_train_if_history_exists=SKIP_TRAIN_IF_HISTORY_EXISTS,
    force_retrain=FORCE_RETRAIN,
    device=DEVICE,
    verbose=True,
)
epochs = figure3_epoch_list(
    combined, num_epochs=NUM_EPOCHS, stride=RENDER_EPOCH_STRIDE
)
print(f"epochs={epochs[:3]}…{epochs[-1]}  n={len(epochs)}")
print(f"combined={repo_relative_path(layout['combined'])}")


Reusing history outputs/figure3_learning_convergence/prediction_history/combined_prediction_history.json
epochs=[0, 25, 50]…200  n=9
combined=outputs/figure3_learning_convergence/prediction_history/combined_prediction_history.json


In [4]:
scenes = generate_figure3_scenes(
    layout=layout,
    repo=ROOT,
    epochs=epochs,
    combined=combined,
    camera_location=CAMERA_LOCATION,
    camera_look_at=CAMERA_LOOK_AT,
    fov_deg=CAMERA_FOV_DEG,
    light_intensity=LIGHT_INTENSITY,
)
print(f"world={repo_relative_path(scenes.scenes_dir / 'figure3_world.inc')}")
print(f"n_pov={len(scenes.pov_paths)}  camera={scenes.camera}")
print(f"includes={scenes.includes}")


world=outputs/figure3_learning_convergence/povray/scenes/figure3_world.inc
n_pov=9  camera={'location': [-15.0, 85.0, 8.0], 'look_at': [-15.1, 0.5, 10.5]}
includes=['figure3_world.inc']


In [5]:
from gummybear_illustration.figure3_pov_scene import figure3_mesh_metrics

add_bear_to_figure3_scenes(scenes, bear_id=MAIN_BEAR_ID)
_, _, span = figure3_mesh_metrics(scenes.bears[MAIN_BEAR_ID]["mesh"])
validation_translate = tuple(s * span for s in VALIDATION_BEAR_SHIFT_SPANS)
training_translate = tuple(s * span for s in TRAINING_BEAR_SHIFT_SPANS)
add_bear_to_figure3_scenes(
    scenes,
    bear_id=VALIDATION_BEAR_ID,
    scale=VALIDATION_BEAR_SCALE,
    translate=validation_translate,
)
add_bear_to_figure3_scenes(
    scenes,
    bear_id=TRAINING_BEAR_ID,
    scale=TRAINING_BEAR_SCALE,
    translate=training_translate,
)
print(f"includes={scenes.includes}")
print(f"bears={list(scenes.bears)}")
print(
    f"validation_bear shift_spans={VALIDATION_BEAR_SHIFT_SPANS} "
    f"translate={validation_translate} scale={VALIDATION_BEAR_SCALE}"
)
print(
    f"training_bear shift_spans={TRAINING_BEAR_SHIFT_SPANS} "
    f"translate={training_translate} scale={TRAINING_BEAR_SCALE}"
)
print(f"bear_mesh={repo_relative_path(scenes.scenes_dir / 'figure3_bear.inc')}")
print(
    f"validation_bear_object="
    f"{repo_relative_path(scenes.scenes_dir / 'figure3_bear_validation_object.inc')}"
)
print(
    f"training_bear_object="
    f"{repo_relative_path(scenes.scenes_dir / 'figure3_bear_train_object.inc')}"
)


includes=['figure3_world.inc', 'figure3_bear.inc', 'figure3_bear_object.inc', 'figure3_bear_validation.inc', 'figure3_bear_validation_object.inc', 'figure3_bear_train.inc', 'figure3_bear_train_object.inc']
bears=['default', 'validation', 'train']
validation_bear shift_spans=(-1.5, 1.5, 0.25) translate=(-20.013140201568604, 20.013140201568604, 3.3355233669281006) scale=0.38
training_bear shift_spans=(-2.3, 1.5, 0.25) translate=(-30.686814975738525, 20.013140201568604, 3.3355233669281006) scale=0.38
bear_mesh=outputs/figure3_learning_convergence/povray/scenes/figure3_bear.inc
validation_bear_object=outputs/figure3_learning_convergence/povray/scenes/figure3_bear_validation_object.inc
training_bear_object=outputs/figure3_learning_convergence/povray/scenes/figure3_bear_train_object.inc


In [6]:
_marker_kw = dict(
    ball_lights=BALL_LIGHTS,
    ball_spotlights=BALL_SPOTLIGHTS,
    ball_spotlight_angle_deg=BALL_SPOTLIGHT_ANGLE_DEG,
    ball_light_intensity=BALL_LIGHT_INTENSITY,
    draw_prediction_links=DRAW_PREDICTION_LINKS,
    distance_luminosity=DISTANCE_LUMINOSITY,
    distance_transparency=DISTANCE_TRANSPARENCY,
    green_luminosity=GREEN_LUMINOSITY,
    pred_luminosity_at_target=PRED_LUMINOSITY_AT_TARGET,
    pooling_luminosity_at_target=POOLING_LUMINOSITY_AT_TARGET,
    fourier_luminosity_at_target=FOURIER_LUMINOSITY_AT_TARGET,
    pred_luminosity_hold_mm=PRED_LUMINOSITY_HOLD_MM,
    pred_luminosity_zero_mm=PRED_LUMINOSITY_ZERO_MM,
    pred_transmit_hold_mm=PRED_TRANSMIT_HOLD_MM,
    pred_transmit_far=PRED_TRANSMIT_FAR,
    pred_transmit_zero_mm=PRED_TRANSMIT_ZERO_MM,
    pred_transmit_at_target=PRED_TRANSMIT_AT_TARGET,
)

add_markers_to_figure3_scenes(
    scenes,
    combined,
    split="test",
    bear_id=MAIN_BEAR_ID,
    **_marker_kw,
)
add_markers_to_figure3_scenes(
    scenes,
    combined,
    split="validation",
    bear_id=VALIDATION_BEAR_ID,
    **_marker_kw,
)
add_markers_to_figure3_scenes(
    scenes,
    combined,
    split="train",
    bear_id=TRAINING_BEAR_ID,
    **_marker_kw,
)
add_loss_board_to_figure3_scenes(
    scenes,
    combined,
    width_mm=BOARD_WIDTH_MM,
    aspect=BOARD_ASPECT,
    thickness_mm=BOARD_THICKNESS_MM,
    center_fwd_frac=BOARD_CENTER_FWD_FRAC,
    center_right_frac=BOARD_CENTER_RIGHT_FRAC,
    center_up_frac=BOARD_CENTER_UP_FRAC,
    label_font_px=BOARD_LABEL_FONT_PX,
    curve_width_px=BOARD_CURVE_WIDTH_PX,
    dot_radius_mm=BOARD_DOT_RADIUS_MM,
)
print(f"marker_layers={scenes.marker_layers}")
print(f"board_layers={scenes.board_layers}")
print(f"example_pov={repo_relative_path(scenes.pov_paths[-1])}")


Writing 9 marker includes with 9 processes (start method=fork).
Writing 9 marker includes with 9 processes (start method=fork).
Writing 9 marker includes with 9 processes (start method=fork).
marker_layers=[{'bear_id': 'default', 'split': 'test', 'template': 'figure3_epoch_{epoch:04d}_markers_default_test.inc'}, {'bear_id': 'validation', 'split': 'validation', 'template': 'figure3_epoch_{epoch:04d}_markers_validation_validation.inc'}, {'bear_id': 'train', 'split': 'train', 'template': 'figure3_epoch_{epoch:04d}_markers_train_train.inc'}]
board_layers=[{'board_id': 'loss', 'template': 'figure3_epoch_{epoch:04d}_board_loss.inc'}]
example_pov=outputs/figure3_learning_convergence/povray/scenes/figure3_epoch_0200.pov


In [7]:
pngs, notes = render_figure3_scenes(
    scenes,
    render=RENDER,
    multiprocessing=RENDER_MULTIPROCESSING,
    gif_duration_ms=GIF_DURATION_MS,
)
for note in notes:
    print(note)
print(f"n_renders={len(pngs)}")
if pngs:
    print(f"final_png={repo_relative_path(layout['final_png'])}")
    print(f"final_gif={repo_relative_path(layout['final_gif'])}")


Rendering 9 POV frames with 9 processes (+WT1 each).


povray: cannot open the user configuration file ~/.povray/3.7/povray.conf: No such file or directory
Persistence of Vision(tm) Ray Tracer Version 3.7.0.10.unofficial (clang++ 21.0.0
 @ aarch64-apple-darwin25.6.0)
This is an unofficial version compiled by:
 Homebrew
 The POV-Ray Team is not responsible for supporting this version.

POV-Ray is based on DKBTrace 2.12 by David K. Buck & Aaron A. Collins
Copyright 1991-2013 Persistence of Vision Raytracer Pty. Ltd.

Primary POV-Ray 3.7 Architects/Developers: (Alphabetically)
  Chris Cason         Thorsten Froehlich  Christoph Lipka   

With Assistance From: (Alphabetically)
  Ton van den Broek   Nicolas Calimet     Jerome Grimbert     James Holsenback  
  Christoph Hormann   Nathan Kopp         Juha Nieminen     

Past Contributors: (Alphabetically)
  Steve Anger         Eric Barish         Dieter Bayer        David K. Buck     
  Nicolas Calimet     Chris Cason         Aaron A. Collins    Chris Dailey      
  Steve Demlow        Andreas Di

n_renders=9
final_png=outputs/figure3_learning_convergence/final/figure3_m8_vs_fourier_convergence.png
final_gif=outputs/figure3_learning_convergence/final/figure3_m8_vs_fourier_convergence.gif


## 2. Summary


In [8]:
combined = load_history(layout["combined"])
sample_id, p_err, f_err, adv = select_best_fourier_advantage_sample(combined)

print("JSON coordinate histories:")
print(f"  {repo_relative_path(layout['pooling'])}")
print(f"  {repo_relative_path(layout['fourier'])}")
print(f"  {repo_relative_path(layout['combined'])}")
print("POV-Ray scene files:")
print(f"  {repo_relative_path(layout['scenes'])}")
print("Rendered figures:")
print(f"  {repo_relative_path(layout['renders'])}")
print(f"  {repo_relative_path(layout['final_png'])}")
print(f"  {repo_relative_path(layout['final_gif'])}")
print(
    "Tracked sample that best demonstrates the Fourier advantage over M8 pooling:"
)
print(
    f"  sample_id={sample_id}  "
    f"final pooled error={p_err:.4f}  "
    f"final Fourier error={f_err:.4f}  "
    f"advantage (pooled−Fourier)={adv:.4f}"
)


JSON coordinate histories:
  outputs/figure3_learning_convergence/prediction_history/m8_pooling_history.json
  outputs/figure3_learning_convergence/prediction_history/fourier_pooling_history.json
  outputs/figure3_learning_convergence/prediction_history/combined_prediction_history.json
POV-Ray scene files:
  outputs/figure3_learning_convergence/povray/scenes
Rendered figures:
  outputs/figure3_learning_convergence/povray/renders
  outputs/figure3_learning_convergence/final/figure3_m8_vs_fourier_convergence.png
  outputs/figure3_learning_convergence/final/figure3_m8_vs_fourier_convergence.gif
Tracked sample that best demonstrates the Fourier advantage over M8 pooling:
  sample_id=bear_m8_high_000028  final pooled error=8.0392  final Fourier error=0.4419  advantage (pooled−Fourier)=7.5973
